# 신규 셀러 온보딩 운영 의사결정 모의실험

## tl;dr

이 노트북은 실제 배민스토어의 병목을 찾는 분석이 아닙니다. 가상 셀러 240곳의 8주 기록으로 완료, 미완료 분리, 재문의 계산, 업무 우선순위 비교가 재현되는지 시험합니다. 실제 단계별 병목, 처리 여력과 정책 효과는 회사 데이터로 다시 검증해야 합니다.


## Context & Methods

### Key Assumptions

- 관찰 기간: 2026-08-11~2026-10-06
- 데이터 단위: 셀러 1곳, 단계 진행 1회, 문의 1건, 지원 업무 1건
- 14일 내 완료율은 신청 후 14일이 지난 셀러만 분모에 포함합니다.
- 정책 비교는 같은 업무 유입과 일일 처리 여력에서 우선순위만 바꿉니다.
- 고정 난수 20261006을 사용해 같은 자료를 다시 만들 수 있습니다.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / 'data'
sellers = pd.read_csv(DATA / 'sellers.csv', parse_dates=['applied_at'])
stages = pd.read_csv(DATA / 'stage_history.csv', parse_dates=['started_at', 'completed_at'])
inquiries = pd.read_csv(DATA / 'inquiries.csv', parse_dates=['created_at', 'first_response_at', 'resolved_at'])
tasks = pd.read_csv(DATA / 'work_tasks.csv')
print(f'셀러 {len(sellers):,}곳 | 단계 기록 {len(stages):,}건 | 문의 {len(inquiries):,}건 | 업무 {len(tasks):,}건')


셀러 240곳 | 단계 기록 1,135건 | 문의 540건 | 업무 1,225건


## Data

### 1. 데이터 품질 확인


In [2]:
checks = {
    '셀러 ID 중복': int(sellers.seller_id.duplicated().sum()),
    '단계 복합키 중복': int(stages.duplicated(['seller_id', 'stage', 'stage_order']).sum()),
    '문의 연결 누락': int((~inquiries.seller_id.isin(sellers.seller_id)).sum()),
    '업무 연결 누락': int((~tasks.seller_id.isin(sellers.seller_id)).sum()),
    '음수 대기시간': int((stages.queue_hours < 0).sum()),
    '완료일 역전': int(((stages.completed_at.notna()) & (stages.completed_at < stages.started_at)).sum()),
}
print(pd.Series(checks, name='실패 행 수').to_string())
assert sum(checks.values()) == 0


셀러 ID 중복     0
단계 복합키 중복    0
문의 연결 누락     0
업무 연결 누락     0
음수 대기시간      0
완료일 역전       0


## Results

### 2. 완료 시간과 단계별 병목


In [3]:
ready = stages[(stages.stage == '판매 준비 완료') & stages.completed_at.notna()][['seller_id', 'completed_at']]
status = sellers.merge(ready, on='seller_id', how='left')
status['lead_days'] = (status.completed_at - status.applied_at).dt.total_seconds() / 86400
mature = status[status.applied_at < pd.Timestamp('2026-09-23')].copy()
mature['within_14d'] = mature.completed_at.notna() & mature.lead_days.le(14)
print(f'완료 {status.completed_at.notna().sum():,}/{len(status):,}곳 ({status.completed_at.notna().mean():.1%})')
print(f'완료 사례 중앙값 {status.lead_days.median():.1f}일 | 90백분위 {status.lead_days.quantile(.9):.1f}일')
print(f'14일 관찰 가능 셀러 완료율 {mature.within_14d.mean():.1%} ({len(mature):,}곳 기준)')
stage_summary = stages[(stages.stage != '신청') & stages.completed_at.notna()].groupby('stage').agg(
    사례=('seller_id', 'count'), 중앙_대기시간=('queue_hours', 'median'),
    p90_대기시간=('queue_hours', lambda s: s.quantile(.9)),
    재작업률=('rework_count', lambda s: (s > 0).mean())
).sort_values('중앙_대기시간', ascending=False)
print()
print(stage_summary.round(2).to_string())


완료 196/240곳 (81.7%)
완료 사례 중앙값 6.0일 | 90백분위 8.2일
14일 관찰 가능 셀러 완료율 89.1% (184곳 기준)

           사례  중앙_대기시간  p90_대기시간  재작업률
stage                                 
상품 등록     205    50.20     72.04  0.23
서류 확인     231    30.90     46.80  0.30
판매 준비 완료  196    27.45     40.45  0.00
계약        219    24.10     36.92  0.00


### 3. 문의가 반복되는 문제


In [4]:
issue_summary = inquiries.groupby('issue_type').agg(
    문의=('inquiry_id', 'count'), 재문의=('repeat_within_72h', 'sum')
).assign(재문의율=lambda frame: frame.재문의 / frame.문의).sort_values(['재문의율', '문의'], ascending=False)
print(issue_summary.head(6).round(3).to_string())
print()
print(f'전체 72시간 내 재문의율: {inquiries.repeat_within_72h.mean():.1%}')


             문의  재문의   재문의율
issue_type                 
사업자 정보       74   18  0.243
서류 보완       114   23  0.202
담당자 확인       42    8  0.190
오픈 일정        19    3  0.158
계약 절차        53    8  0.151
검토 일정        92   12  0.130

전체 72시간 내 재문의율: 15.2%


### 4. 같은 조건에서 우선순위 정책 비교


In [5]:
import json
summary = json.loads((PROJECT_ROOT / 'analysis' / 'summary.json').read_text(encoding='utf-8'))
policies = pd.DataFrame(summary['policy_metrics'])[[
    'policy', 'completion_rate', 'median_wait_days', 'p90_wait_days',
    'sla_3day_rate', 'ready_stage_sla_3day_rate', 'max_wait_unresolved_days'
]]
print(policies.to_string(index=False))


      policy  completion_rate  median_wait_days  p90_wait_days  sla_3day_rate  ready_stage_sla_3day_rate  max_wait_unresolved_days
       접수 순서            0.866               6.0            7.0          0.224                       0.20                         6
오래 대기, 재문의 우선            0.862               5.0           10.0          0.396                       0.35                         9
 판매 준비 임박 우선            0.842               0.0           22.0          0.778                       1.00                        31


## Takeaways

- 가상 자료에서 상품 등록 단계의 중앙 대기시간이 가장 길어, 첫 개선 대상으로 삼을 근거가 생겼습니다.
- 판매 준비 임박 업무만 먼저 처리하면 해당 구간의 3일 이내 처리율은 높아지지만, 오래 남는 업무가 크게 늘어나는 부작용이 있습니다.
- 따라서 기본 운영안은 ‘오래 대기, 재문의 우선’으로 두고, 판매 준비 임박 업무는 제한된 별도 처리 구간으로 운영하는 혼합 정책을 제안합니다.
- 다음 검증은 실제 운영 자료가 확보됐을 때 단계 정의, 처리 여력, 문의 분류 기준을 다시 맞추는 것입니다.
